In [12]:
import snappy #only necessary if running in sage
import os #used to add lines separating each manifold entry
from sympy import nsimplify #library used to simplify decimal angle to a ratio of pi

rational_estimation_tolerance = 1e-3 #level of accuracy when converting the angle to a ratio of pi

term_size = os.get_terminal_size() #size of the output terminal for printing line
S = snappy.twister.Surface('S_1_1') #genus 1 surface with one puncture

for i in range(1,2):
    M = S.bundle(f'a*(B)^{i}') #construct M as a surface bundle RL^i
    
    #print info on each manifold
    print(f'\nNumber of Tetrahedra = {i+1}\n\n{M} \nVolume = {M.volume()}')
    iterator = 0
    
    #loop over each tetrahedron
    for tet in M.tetrahedra_shapes():
        
        #set values which will be printed
        shape = tet['rect'] #shape parameter for the current tet.
        shapep = 1-1/(shape) #another shape parameter obtained via 1-1/z
        shapepp = 1/(1-shape) #the third shape parameter obtained via 1/(1-z)
        
        #aruments of the shape parameters
        angle = arg(shape)
        anglep = arg(shapep) 
        anglepp = arg(shapepp)
        
        #print the volume of each individual tetrahedron
        volume = tet['volume']
        
        #print separating line between tetrahedra
        print('_' * term_size.columns)
        
        #print tetrahedra index
        print(f'\nTetrahedra {iterator}')
              
        #print shape parameters
        print(f'Volume = {volume} \n')
        print(f'z = {shape} \nz\' = {shapep}')
        print(f'z\'\' = {shapepp} \n')
              
        #print the angles as decimal approximations      
        print(f'\u03B8 = {angle}')
        print(f'\u03B8\' = {anglep}')
        print(f'\u03B8\'\' = {anglepp}')
        
        #estimate the angle as a ratio of pi
        print(f'\n\u03C6 = {pi*nsimplify(angle/pi, tolerance=rational_estimation_tolerance)}')
        print(f'\u03C6\' = {pi*nsimplify(anglep/pi, tolerance=rational_estimation_tolerance)}')
        print(f'\u03C6\'\' = {pi*nsimplify(anglepp/pi, tolerance=rational_estimation_tolerance)}\n')
        iterator += 1
    
    print('=' * term_size.columns, '\n')


Number of Tetrahedra = 2

a*(B)^1(0,0) 
Volume = 2.02988321281931
________________________________________________________________________________

Tetrahedra 0
Volume = 1.01494160641 

z = 0.500000000000000 + 0.866025403784439*I 
z' = 0.500000000000000 + 0.866025403784439*I
z'' = 0.500000000000000 + 0.866025403784438*I 

θ = 1.04719755119660
θ' = 1.04719755119660
θ'' = 1.04719755119660

φ = 1/3*pi
φ' = 1/3*pi
φ'' = 1/3*pi

________________________________________________________________________________

Tetrahedra 1
Volume = 1.01494160641 

z = 0.500000000000000 + 0.866025403784439*I 
z' = 0.500000000000000 + 0.866025403784439*I
z'' = 0.500000000000000 + 0.866025403784438*I 

θ = 1.04719755119660
θ' = 1.04719755119660
θ'' = 1.04719755119660

φ = 1/3*pi
φ' = 1/3*pi
φ'' = 1/3*pi




In [60]:
import snappy 
colormaps=sage.plot.colors.colormaps.keys()
class InspectTets():
    def __init__(self, rng = 100):
        
        colormaps=sage.plot.colors.colormaps.keys()
        S = snappy.twister.Surface('S_1_1') #genus 1 surface with one puncture

        point_tot_vol = []
        point_min_vol = []
        point_max_vol = []
        point_avg_vol = []
        for i in range(1,rng):
            M = S.bundle(f'a*(B)^{i}') #construct M as a surface bundle RL^i
            volume = M.volume()

            pos_vol = [] #snappy seems to return negative volumes for some tetrahedra. We will force them to be positive here.
            for vol in M.tetrahedra_shapes('volume'):
                pos_vol.append(abs(vol))

            point_tot_vol.append((i+1, volume))
            point_min_vol.append((i+1, min(pos_vol)))
            point_max_vol.append((i+1, max(pos_vol)))
            point_avg_vol.append((i+1, volume/(i+1)))
            
            
        self.point_tot_vol = point_tot_vol
        self.point_min_vol = point_min_vol
        self.point_max_vol = point_max_vol
        self.point_avg_vol = point_avg_vol
        self.rng = rng
        
        self.plots = (
        [list_plot(point_tot_vol, color='blue'),
        list_plot(point_min_vol, color = 'red'),
        list_plot(point_max_vol, color = 'green'),
        list_plot(point_avg_vol, color = 'black')])


In [64]:
tets = InspectTets(1000) #don't run this unless you want to wait a while

In [65]:
import pickle

with open('inspect_tets.pkl', 'wb') as outp:
    tetrahedra = tets
    pickle.dump(tets, outp, pickle.HIGHEST_PROTOCOL)


In [83]:
import pickle

#the data takes a long time to generate, so instead we exported the data for up to 1000 twists to an external file, and import it here
with open('inspect_tets.pkl', 'rb') as inp:
    tets = pickle.load(inp)

In [94]:
import ipywidgets as widgets

@interact(layout = {'left' : ['checktot''checkmin''checkmax','checkavg'], 'right' : ['colortot','colormin','colormax','coloravg'], 'bottom' : ['zoom']})
def myplot(checktot = checkbox(True, label = 'Total Volume'),
          checkmin = checkbox(True, label = 'Minimum Volume'),
          checkmax = checkbox(True, label = 'Maximum Volume'),
          checkavg = checkbox(True, label = 'Average Volume'),
          colortot = (color_selector(widget='colorpicker', label="Total Volume", default = 'black')),
          colormin = (color_selector(widget='colorpicker', label="Minimum Volume", default = 'purple')),
          colormax = (color_selector(widget='colorpicker', label="Maximum Volume", default = 'red')),
          coloravg = (color_selector(widget='colorpicker', label="Average Volume", default = 'green')),
          zoom=widgets.IntRangeSlider(value = [0,1000], min = 0, max = 1000, step = 1),
          auto_update=False):
    plots = False
    if checktot:
        plots += list_plot(tets.point_tot_vol[zoom[0]:zoom[1]], color = colortot)
    if checkmin:
        plots += list_plot(tets.point_min_vol[zoom[0]:zoom[1]], color = colormin)
    if checkmax:
        plots += list_plot(tets.point_max_vol[zoom[0]:zoom[1]], color = colormax)
    if checkavg:
        plots += list_plot(tets.point_avg_vol[zoom[0]:zoom[1]], color = coloravg)

    show (plots)

Manual interactive function <function myplot at 0x7f61318a6af0> with 9 widgets
  checktot: Checkbox(value=True, description='Total Volume')
  checkmin: Checkbox(value=True, description='Minimum Volume')
  checkmax: Checkbox(value=True, description='Maximum Volume')
  checkavg: Checkbox(value=True, description='Average Volume')
  colortot: SageColorPicker(value='#000000', description='Total Volume')
  colormin: SageColorPicker(value='#800080', description='Minimum Volume')
  colormax: SageColorPicker(value='#ff0000', description='Maximum Volume')
  coloravg: SageColorPicker(value='#008000', description='Average Volume')
  zoom: IntRangeSlider(value=(0, 1000), description='zoom', max=1000)

In [92]:
# Imports for JupyterLite
try:
    import piplite
    await piplite.install(['ipywidgets'])
except ImportError:
    pass

In [93]:
import ipywidgets as widgets